# documents

> Document list, detail, and delete routes

In [ ]:
#| default_exp routes.documents

In [ ]:
#| export
from typing import Callable, Dict, List, Tuple

from fasthtml.common import Div, P
from cjm_fasthtml_app_core.core.routing import APIRouter

from cjm_transcript_workflow_management.services.management import ManagementService
from cjm_transcript_workflow_management.models import ManagementUrls
from cjm_transcript_workflow_management.components.document_detail import (
    render_document_detail, render_detail_error
)
from cjm_transcript_workflow_management.html_ids import ManagementHtmlIds
from cjm_transcript_workflow_management.routes.core import DEBUG_MANAGEMENT_ROUTES

from cjm_fasthtml_daisyui.components.feedback.alert import alert_colors

## Router Initialization

In [ ]:
#| export
def init_document_router(
    service:ManagementService,  # Service for graph queries
    prefix:str,  # Route prefix (e.g., "/manage/documents")
    urls:ManagementUrls,  # URL bundle (populated after init)
    refresh_items:Callable,  # async () -> refresh items from service
    refresh_items_oob:Callable,  # async () -> refresh + targeted VC OOB tuple
    render_list:Callable,  # () -> rendered document list component
    render_page:Callable,  # () -> rendered full management page
    get_selected_ids:Callable,  # () -> List[str] of selected document IDs
) -> Tuple[APIRouter, Dict[str, Callable]]:  # (router, routes dict)
    """Initialize document list, detail, and delete routes."""
    router = APIRouter(prefix=prefix)
    routes = {}
    
    @router
    async def management_page(request):
        """Return the full management page (header + import + list)."""
        if DEBUG_MANAGEMENT_ROUTES:
            print("[ROUTES] management_page called")
        await refresh_items()
        return render_page()
    
    routes["management_page"] = management_page
    
    @router
    async def list_documents(request):
        """Return rendered document list."""
        if DEBUG_MANAGEMENT_ROUTES:
            print("[ROUTES] list_documents called")
        await refresh_items()
        return render_list()
    
    routes["list_documents"] = list_documents
    
    @router
    async def document_detail(request, doc_id:str=""):
        """Return document detail dashboard."""
        if DEBUG_MANAGEMENT_ROUTES:
            print(f"[ROUTES] document_detail called for {doc_id}")
        
        if not doc_id:
            return render_detail_error("No document ID provided.", urls=urls)
        
        detail = await service.get_document_detail_async(doc_id)
        
        if detail is None:
            return render_detail_error(
                f"Document {doc_id[:12]}... not found.", urls=urls
            )
        
        return render_document_detail(detail, urls)
    
    routes["document_detail"] = document_detail
    
    @router.post
    async def delete_document(request):
        """Delete a single document and return refreshed content."""
        form_data = await request.form()
        doc_id = form_data.get("doc_id", "")
        return_page = form_data.get("return_page", "")
        
        if DEBUG_MANAGEMENT_ROUTES:
            print(f"[ROUTES] delete_document: {doc_id}, return_page={return_page}")
        
        if doc_id:
            await service.delete_document_async(doc_id)
        
        # From detail view: full page re-render (back to list)
        if return_page:
            await refresh_items()
            return render_page()
        
        # From list view: targeted VC OOB updates
        return await refresh_items_oob()
    
    routes["delete_document"] = delete_document
    
    @router.post
    async def delete_selected(request):
        """Delete selected documents and return targeted VC OOB updates."""
        doc_ids = get_selected_ids()
        
        if DEBUG_MANAGEMENT_ROUTES:
            print(f"[ROUTES] delete_selected: {len(doc_ids)} docs")
        
        if doc_ids:
            await service.delete_documents_async(doc_ids)
        
        return await refresh_items_oob()
    
    routes["delete_selected"] = delete_selected
    
    return router, routes

In [ ]:
assert init_document_router is not None
print("routes.documents imports OK")

routes.documents imports OK


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()